In [ ]:
# ============================================================
# BEDLOCATION: Generate BEDLOCATION.csv from MIMIC-IV transfers
# ============================================================

import os
import pandas as pd

input_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_dir, exist_ok=True)

transfers_path = os.path.join(input_dir, "hosp_transfers.csv")
transfers = pd.read_csv(transfers_path)

print("✅ hosp_transfers.csv loaded:", transfers.shape)
print(transfers.head())

✅ hosp_transfers.csv loaded: (2413581, 7)
   subject_id     hadm_id  transfer_id  eventtype              careunit  \
0    10000032  22595853.0     33258284         ED  Emergency Department   
1    10000032  22595853.0     35223874      admit            Transplant   
2    10000032  22595853.0     36904543  discharge               UNKNOWN   
3    10000032  22841357.0     34100253  discharge               UNKNOWN   
4    10000032  22841357.0     34703856      admit            Transplant   

                intime              outtime  
0  2180-05-06 19:17:00  2180-05-06 23:30:00  
1  2180-05-06 23:30:00  2180-05-07 17:21:27  
2  2180-05-07 17:21:27                  NaN  
3  2180-06-27 18:49:12                  NaN  
4  2180-06-26 21:31:00  2180-06-27 18:49:12  


In [ ]:
bedlocation = transfers[[
    "subject_id", "hadm_id", "careunit", "intime", "outtime"
]].rename(columns={
    "subject_id": "pat_id",
    "hadm_id": "csn",
    "careunit": "bed_unit",
    "intime": "bed_location_start",
    "outtime": "bed_location_end"
})

print("✅ BEDLOCATION subset created:", bedlocation.shape)
print(bedlocation.head())

✅ BEDLOCATION subset created: (2413581, 5)
     pat_id         csn              bed_unit   bed_location_start  \
0  10000032  22595853.0  Emergency Department  2180-05-06 19:17:00   
1  10000032  22595853.0            Transplant  2180-05-06 23:30:00   
2  10000032  22595853.0               UNKNOWN  2180-05-07 17:21:27   
3  10000032  22841357.0               UNKNOWN  2180-06-27 18:49:12   
4  10000032  22841357.0            Transplant  2180-06-26 21:31:00   

      bed_location_end  
0  2180-05-06 23:30:00  
1  2180-05-07 17:21:27  
2                  NaN  
3                  NaN  
4  2180-06-27 18:49:12  


In [ ]:
bedlocation["bed_unit"] = bedlocation["bed_unit"].fillna("Not Recorded")

bedlocation["bed_location_start"] = pd.to_datetime(
    bedlocation["bed_location_start"], errors="coerce"
)
bedlocation["bed_location_end"] = pd.to_datetime(
    bedlocation["bed_location_end"], errors="coerce"
)

print("✅ After cleaning:", bedlocation.shape)
print(bedlocation.isna().sum())

✅ After cleaning: (2413581, 5)
pat_id                     0
csn                   408977
bed_unit                   0
bed_location_start         0
bed_location_end      546123
dtype: int64


In [ ]:
out_path = os.path.join(output_dir, "BEDLOCATION.csv")
bedlocation.to_csv(out_path, index=False)

print(f"✅ BEDLOCATION.csv saved to {out_path} with shape {bedlocation.shape}")
print("Final columns:", bedlocation.columns.tolist())
print(bedlocation.head())

✅ BEDLOCATION.csv saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/BEDLOCATION.csv with shape (2413581, 5)
Final columns: ['pat_id', 'csn', 'bed_unit', 'bed_location_start', 'bed_location_end']
     pat_id         csn              bed_unit  bed_location_start  \
0  10000032  22595853.0  Emergency Department 2180-05-06 19:17:00   
1  10000032  22595853.0            Transplant 2180-05-06 23:30:00   
2  10000032  22595853.0               UNKNOWN 2180-05-07 17:21:27   
3  10000032  22841357.0               UNKNOWN 2180-06-27 18:49:12   
4  10000032  22841357.0            Transplant 2180-06-26 21:31:00   

     bed_location_end  
0 2180-05-06 23:30:00  
1 2180-05-07 17:21:27  
2                 NaT  
3                 NaT  
4 2180-06-27 18:49:12  
